In [1]:
import pandas as pd
import numpy as np
import random
from pathlib import Path

RAW_PATH = Path('/home/duong/RS/raw_data/2019-Nov.csv')
CLEAN_DIR = Path('/home/duong/RS/cleaned_data')
SEED = 42

In [2]:
p = 0.10   # fraction of rows kept (same row-level sampling as before)
rng = np.random.default_rng(SEED)

dtypes = {
    'event_type': 'category', 'product_id': 'int32', 'category_id': 'int64',
    'category_code': 'str', 'brand': 'str', 'price': 'float32',
    'user_id': 'int32', 'user_session': 'str',
}

parts = []
n_total = 0
for chunk in pd.read_csv(RAW_PATH, dtype=dtypes, chunksize=2_000_000):
    n_total += len(chunk)
    chunk = chunk[rng.random(len(chunk)) < p].copy()
    chunk['event_time'] = pd.to_datetime(chunk['event_time'], utc=True).dt.tz_localize(None)
    # user_session is only ever used as a grouping key -> store a 64-bit hash instead of a 36-char uuid
    chunk['user_session'] = pd.util.hash_pandas_object(chunk['user_session'], index=False).astype('int64')
    parts.append(chunk)

df = pd.concat(parts, ignore_index=True)
del parts
print(f'{n_total:,} raw rows read -> {len(df):,} kept ({len(df)/n_total:.1%})')

67,501,979 raw rows read -> 6,752,985 kept (10.0%)


In [3]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:01,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.270004,558856683,-7486665958022197604
1,2019-11-01 00:00:07,view,5100503,2053013553375346967,NaN,xiaomi,22.680000,520037415,-1432992388141792685
2,2019-11-01 00:00:10,view,1005115,2053013555631882655,electronics.smartphone,apple,949.469971,562079864,6575112487747071954
3,2019-11-01 00:00:17,view,1800729,2053013554415534427,electronics.video.tv,samsung,289.329987,515782589,-1917518832555405868
4,2019-11-01 00:00:23,view,12300928,2053013556311359947,construction.tools.drill,dewalt,98.510002,517081324,5550946264236824808


In [4]:
df['event_type'] = df['event_type'].astype('category') # ep kieu sang category
df['price'] = df['price'].astype('float32')
df['user_id'] = df['user_id'].astype('int32')
df['product_id'] = df['product_id'].astype('int32')
df['category_id'] = df['category_id'].astype('category')
df['event_time'] = pd.to_datetime(df['event_time']).dt.tz_localize(None)
df['hour'] = df['event_time'].dt.hour.astype('int8')
df['day_of_week'] = df['event_time'].dt.dayofweek.astype('int8')
df['brand'] = df['brand'].fillna('unknown').astype('category')
df['category_code'] = df['category_code'].fillna('unknown')

category_split = df['category_code'].str.split('.', n=1, expand=True)
df['main_category'] = category_split[0].astype('category')
df['sub_category'] = category_split[1].fillna('unknown').astype('category')

df.drop(columns=['category_code'], inplace=True)

df = df[df['price'] > 0]

event_weights = {
    'view': 1,
    'cart': 3,
    'purchase': 5
}
df['interaction_score'] = df['event_type'].map(event_weights).astype('int8')

df.reset_index(drop=True, inplace=True)

In [5]:
df.head()

,event_time,event_type,product_id,category_id,brand,price,user_id,user_session,hour,day_of_week,main_category,sub_category,interaction_score
0,2019-11-01 00:00:01,view,1004775,2053013555631882655,xiaomi,183.270004,558856683,-7486665958022197604,0,4,electronics,smartphone,1
1,2019-11-01 00:00:07,view,5100503,2053013553375346967,xiaomi,22.680000,520037415,-1432992388141792685,0,4,unknown,unknown,1
2,2019-11-01 00:00:10,view,1005115,2053013555631882655,apple,949.469971,562079864,6575112487747071954,0,4,electronics,smartphone,1
3,2019-11-01 00:00:17,view,1800729,2053013554415534427,samsung,289.329987,515782589,-1917518832555405868,0,4,electronics,video.tv,1
4,2019-11-01 00:00:23,view,12300928,2053013556311359947,dewalt,98.510002,517081324,5550946264236824808,0,4,construction,tools.drill,1


In [6]:
df['user_session'] = df['user_session'].astype('category').cat.codes


In [7]:
df.sort_values(by=['user_id', 'event_time'], inplace=True)
df.reset_index(drop=True, inplace=True)


In [8]:
df['price'] = df['price'].round(2)
pd.options.display.float_format = '{:.2f}'.format


In [9]:
# Save cleaned data (all users, before the 5-200 interactions filter) -> needed for cold-start / guest modules
CLEAN_DIR.mkdir(exist_ok=True)
df.to_parquet(CLEAN_DIR / 'events_clean_all.parquet', index=False)
print(f'events_clean_all.parquet: {len(df):,} rows')

events_clean_all.parquet: 6,734,076 rows


In [10]:
user_interaction_counts = df['user_id'].value_counts()

stats = user_interaction_counts.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

print(stats.apply(lambda x: f"{x:,.0f}"))

count    1,830,139
mean             4
std              6
min              1
25%              1
50%              2
75%              4
90%              8
95%             12
max          2,344
Name: count, dtype: str


In [11]:
user_counts = df['user_id'].value_counts()

valid_users = user_counts[(user_counts >= 5) & (user_counts <= 200)].index

df_filtered = df[df['user_id'].isin(valid_users)]

df_filtered.reset_index(drop=True, inplace=True)

In [12]:
print(len(df_filtered))
print(len(valid_users))

4192153
394880


In [13]:
df_filtered.head()

,event_time,event_type,product_id,category_id,brand,price,user_id,user_session,hour,day_of_week,main_category,sub_category,interaction_score
0,2019-11-18 12:13:18,view,1002544,2053013555631882655,apple,493.84,120701478,3752546,12,0,electronics,smartphone,1
1,2019-11-19 19:30:34,view,1002544,2053013555631882655,apple,488.82,120701478,3752546,19,1,electronics,smartphone,1
2,2019-11-20 04:25:28,view,1002544,2053013555631882655,apple,488.82,120701478,3752546,4,2,electronics,smartphone,1
3,2019-11-21 14:56:03,view,1002544,2053013555631882655,apple,488.69,120701478,3752546,14,3,electronics,smartphone,1
4,2019-11-23 05:30:25,view,1002544,2053013555631882655,apple,487.53,120701478,3752546,5,5,electronics,smartphone,1


In [14]:
df_filtered = df_filtered.copy()

sub_split = df_filtered['sub_category'].astype(str).str.split('.', n=1, expand=True)

df_filtered['sub_category'] = sub_split[0].astype('category')

if sub_split.shape[1] > 1:
    df_filtered['item_type'] = sub_split[1].fillna('unknown').astype('category')
else:
    df_filtered['item_type'] = 'unknown'
    df_filtered['item_type'] = df_filtered['item_type'].astype('category')

df_filtered.head()

,event_time,event_type,product_id,category_id,brand,price,user_id,user_session,hour,day_of_week,main_category,sub_category,interaction_score,item_type
0,2019-11-18 12:13:18,view,1002544,2053013555631882655,apple,493.84,120701478,3752546,12,0,electronics,smartphone,1,unknown
1,2019-11-19 19:30:34,view,1002544,2053013555631882655,apple,488.82,120701478,3752546,19,1,electronics,smartphone,1,unknown
2,2019-11-20 04:25:28,view,1002544,2053013555631882655,apple,488.82,120701478,3752546,4,2,electronics,smartphone,1,unknown
3,2019-11-21 14:56:03,view,1002544,2053013555631882655,apple,488.69,120701478,3752546,14,3,electronics,smartphone,1,unknown
4,2019-11-23 05:30:25,view,1002544,2053013555631882655,apple,487.53,120701478,3752546,5,5,electronics,smartphone,1,unknown


In [15]:
# Save the filtered modelling set (users with 5-200 interactions)
df_filtered.to_parquet(CLEAN_DIR / 'events_filtered.parquet', index=False)
print(f'events_filtered.parquet: {len(df_filtered):,} rows, {df_filtered.user_id.nunique():,} users, {df_filtered.product_id.nunique():,} items')

events_filtered.parquet: 4,192,153 rows, 394,880 users, 135,842 items


In [16]:
# ---- Recommender-oriented EDA summary (filtered set) ----
n_users, n_items = df_filtered.user_id.nunique(), df_filtered.product_id.nunique()
print('time range      :', df_filtered.event_time.min(), '->', df_filtered.event_time.max())
print('events / users / items:', f'{len(df_filtered):,} / {n_users:,} / {n_items:,}')
print('\nevent mix (%):'); print((df_filtered.event_type.value_counts(normalize=True)*100).round(2).to_string())

ui = df_filtered.groupby(['user_id', 'product_id'], observed=True).size()
print(f'\nunique (user,item) pairs: {len(ui):,}   matrix density: {len(ui)/(n_users*n_items):.2e}   sparsity: {1-len(ui)/(n_users*n_items):.6f}')
print('pairs interacted with >1 event: %.1f%%' % ((ui > 1).mean()*100))

print('\nmissing-info share (%):')
print(pd.Series({
    'brand == unknown': (df_filtered.brand == 'unknown').mean(),
    'main_category == unknown': (df_filtered.main_category == 'unknown').mean(),
    'item_type == unknown': (df_filtered.item_type == 'unknown').mean(),
}).mul(100).round(1).to_string())

ipop = df_filtered.groupby('product_id').size().sort_values(ascending=False)
print(f'\nitems with <5 interactions: {(ipop<5).mean():.1%}   top 1% of items = {ipop.head(max(1,len(ipop)//100)).sum()/ipop.sum():.1%} of interactions   top 20%: {ipop.head(len(ipop)//5).sum()/ipop.sum():.1%}')

pur = df_filtered[df_filtered.event_type == 'purchase']
print(f'\npurchases: {len(pur):,} by {pur.user_id.nunique():,} users ({pur.user_id.nunique()/n_users:.1%} of users); distinct purchased items: {pur.product_id.nunique():,}')
sess_pur = pur.groupby('user_session').product_id.nunique()
print(f'sessions with a purchase: {len(sess_pur):,}; with >=2 distinct purchased items (basket-able): {(sess_pur>=2).sum():,} ({(sess_pur>=2).mean():.1%})')
daily = df_filtered.groupby(df_filtered.event_time.dt.date).size()
print('\ndaily events: min %s  median %s  max %s' % (f'{daily.min():,}', f'{int(daily.median()):,}', f'{daily.max():,}'))
print('top-10 brands:', ', '.join(df_filtered[df_filtered.brand != 'unknown'].brand.value_counts().head(10).index.astype(str)))
print('main categories (%):'); print((df_filtered.main_category.value_counts(normalize=True)*100).round(1).head(8).to_string())

time range      : 2019-11-01 00:00:17 -> 2019-11-30 23:59:52
events / users / items: 4,192,153 / 394,880 / 135,842

event mix (%):
event_type
view       94.29
cart        4.45
purchase    1.26



unique (user,item) pairs: 3,437,598   matrix density: 6.41e-05   sparsity: 0.999936
pairs interacted with >1 event: 14.8%

missing-info share (%):
brand == unknown           14.10
main_category == unknown   32.80
item_type == unknown       70.70

items with <5 interactions: 46.6%   top 1% of items = 41.2% of interactions   top 20%: 86.1%

purchases: 52,825 by 39,406 users (10.0% of users); distinct purchased items: 13,575
sessions with a purchase: 51,077; with >=2 distinct purchased items (basket-able): 1,081 (2.1%)



daily events: min 79,327  median 98,284  max 473,200
top-10 brands: samsung, apple, xiaomi, huawei, lucente, lg, bosch, respect, lenovo, sony
main categories (%):
main_category
unknown        32.80
electronics    32.80
appliances     13.50
computers       7.00
apparel         4.50
furniture       3.40
auto            1.80
construction    1.80


In [17]:
import plotly.express as px
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
# SQL: SELECT user_id, COUNT(*) as interactions FROM df_filtered GROUP BY user_id
user_activity = df_filtered.groupby('user_id').size().reset_index(name='interactions')

def categorize_user(x):
    if x < 10: return 'Cold User (< 10)'
    elif x < 50: return 'Warm User (10 - 50)'
    else: return 'Power User (> 50)'
    
user_activity['user_type'] = user_activity['interactions'].apply(categorize_user)
user_type_dist = user_activity['user_type'].value_counts().reset_index()
user_type_dist.columns = ['user_type', 'count']

fig = px.pie(user_type_dist, values='count', names='user_type', hole=0.4, title='Phân bổ tệp người dùng')
fig.show(renderer="iframe")

In [18]:
import plotly.express as px

# Tính toán tỷ lệ chuyển đổi
category_funnel = df_filtered.groupby(['main_category', 'event_type'], observed=True).size().unstack(fill_value=0)
category_funnel['conversion_rate'] = (category_funnel['purchase'] / category_funnel['view']) * 100

# Lọc các ngành hàng có đủ dữ liệu (>1000 view) và lấy top 10
top_converting = category_funnel[category_funnel['view'] > 1000].sort_values(by='conversion_rate', ascending=False).head(10).reset_index()

# Trực quan hóa
fig_funnel = px.bar(
    top_converting, 
    x='main_category', 
    y='conversion_rate',
    title='Top 10 ngành hàng có tỷ lệ chuyển đổi cao nhất',
    labels={'conversion_rate': 'Tỷ lệ chuyển đổi (%)', 'main_category': 'Ngành hàng chính'},
    color='conversion_rate',
    color_continuous_scale='Viridis'
)

fig_funnel.show(renderer="iframe")

In [19]:
import plotly.express as px
from itertools import combinations
from collections import Counter
import pandas as pd

# Lọc các phiên có > 1 sản phẩm
session_items = df_filtered.groupby('user_session')['product_id'].nunique()
multi_item_sessions = session_items[session_items > 1].index

basket_data = df_filtered[df_filtered['user_session'].isin(multi_item_sessions)]
basket_grouped = basket_data.groupby('user_session')['brand'].unique()

pair_counts = Counter()
for brands in basket_grouped:
    valid_brands = [b for b in brands if b != 'unknown']
    if len(valid_brands) > 1:
        for pair in combinations(sorted(valid_brands), 2):
            pair_counts[pair] += 1

# Chuyển kết quả Top 10 sang DataFrame để vẽ biểu đồ
top_10_pairs = pair_counts.most_common(10)
pair_df = pd.DataFrame([
    {'Cặp thương hiệu': f"{pair[0]} & {pair[1]}", 'Tần suất': count} 
    for pair, count in top_10_pairs
])

# Trực quan hóa
fig_basket = px.bar(
    pair_df,
    x='Tần suất',
    y='Cặp thương hiệu',
    orientation='h',
    title='Top 10 cặp thương hiệu thường được tương tác trong cùng 1 phiên',
    text='Tần suất'
)
fig_basket.update_yaxes(autorange="reversed") # Đảo ngược trục Y để Top 1 nằm trên cùng
fig_basket.show(renderer="iframe")

In [20]:
import plotly.express as px

# Lọc hành vi cart và purchase
action_df = df_filtered[df_filtered['event_type'].isin(['cart', 'purchase'])]

# Tạo ma trận Pivot
time_context = action_df.pivot_table(
    index='hour', 
    columns='main_category', 
    values='interaction_score', 
    aggfunc='count', 
    fill_value=0,
    observed=True
)

# Trực quan hóa bằng Heatmap
fig_time = px.imshow(
    time_context,
    labels=dict(x="Ngành hàng", y="Khung giờ (Hour)", color="Số lượt tương tác"),
    x=time_context.columns,
    y=time_context.index,
    title='Cường độ mua sắm theo khung giờ trong ngày và ngành hàng',
    aspect="auto",
    color_continuous_scale='YlGnBu'
    
)

# Chỉnh trục Y để hiển thị đủ 24 mốc giờ
fig_time.update_yaxes(tickmode='linear', dtick=1) 
fig_time.show(renderer="iframe")

In [21]:
import plotly.express as px

# Tính tổng tương tác của từng sản phẩm và sắp xếp giảm dần
item_popularity = df_filtered.groupby('product_id').size().sort_values(ascending=False).reset_index(name='interactions')

# Reset index để tạo trục X (Thứ hạng phổ biến)
item_popularity = item_popularity.reset_index()

fig_tail = px.line(
    item_popularity, 
    x='index', 
    y='interactions', 
    title='Phân phối Long-Tail của Sản phẩm (Độ phổ biến)',
    labels={'index': 'Thứ hạng sản phẩm (Từ phổ biến nhất -> Ngách)', 'interactions': 'Tổng lượt tương tác'}
)

# Ẩn các nhãn trục X vì số thứ tự chi tiết không quan trọng bằng hình dáng đường cong
fig_tail.update_xaxes(showticklabels=False)
fig_tail.show(renderer="iframe")

In [22]:
import plotly.express as px

# Đếm số lượt tương tác trong mỗi phiên
session_length = df_filtered.groupby('user_session').size().value_counts().reset_index()
session_length.columns = ['session_length', 'session_count']

# Chỉ lấy các phiên có từ 1 đến 20 tương tác để biểu đồ không bị kéo dài bởi outlier
session_length = session_length[session_length['session_length'] <= 20].sort_values('session_length')

fig_session = px.bar(
    session_length, 
    x='session_length', 
    y='session_count',
    title='Độ sâu phiên mua sắm (Session Depth)',
    labels={'session_length': 'Số lượt tương tác trong 1 phiên', 'session_count': 'Số lượng phiên'},
    text_auto='.2s'
)

# Cấu hình trục X hiển thị từng số nguyên liên tục
fig_session.update_xaxes(tickmode='linear', dtick=1)
fig_session.show(renderer="iframe")

In [23]:
import plotly.express as px

# Lấy mẫu ngẫu nhiên 100,000 dòng để tránh quá tải trình duyệt khi vẽ Box Plot
df_sample = df_filtered.sample(n=100000, random_state=42)

# Cắt bỏ top 5% giá trị cao nhất (Outliers) để biểu đồ dễ đọc
max_price_threshold = df_sample['price'].quantile(0.95)
df_plot = df_sample[df_sample['price'] <= max_price_threshold]

fig_price = px.box(
    df_plot, 
    x='event_type', 
    y='price', 
    color='event_type',
    title='Mức độ nhạy cảm về giá giữa Hành vi Xem và Mua',
    labels={'event_type': 'Loại hành vi', 'price': 'Giá sản phẩm ($)'}
)

fig_price.show(renderer="iframe")

In [24]:
import pandas as pd

# 1. Kiểm tra tập user đã mua hàng có tồn tại không
purchasers = df_filtered[df_filtered['event_type'] == 'purchase']['user_id'].unique()
print(f"1. Tổng số khách hàng có phát sinh giao dịch mua: {len(purchasers)}")

# 2. Kiểm tra tính toàn vẹn của item_type trong các sự kiện mua
purchase_events = df_filtered[df_filtered['event_type'] == 'purchase']
missing_category = purchase_events['item_type'].isin(['unknown', 'NaN', None, '']).sum()
print(f"2. Số giao dịch mua bị khuyết thông tin item_type: {missing_category} / {len(purchase_events)}")

# 3. Truy vết chéo một khách hàng cụ thể để xem hành trình thực tế
if len(purchasers) > 0:
    sample_uid = purchasers[0] # Lấy người đầu tiên trong danh sách
    print(f"\n3. Lịch sử thô của User {sample_uid}:")
    
    # In ra toàn bộ chuỗi sự kiện của người này theo thời gian
    user_journey = df_filtered[df_filtered['user_id'] == sample_uid][['event_time', 'event_type', 'product_id', 'item_type']]
    print(user_journey.sort_values('event_time').to_string())
    
    # 4. Thuật toán kiểm tra sự giao thoa (Overlap)
    viewed_types = set(user_journey[user_journey['event_type'] == 'view']['item_type'])
    purchased_types = set(user_journey[user_journey['event_type'] == 'purchase']['item_type'])
    overlap = viewed_types.intersection(purchased_types)
    
    # Bỏ qua 'unknown' khi tính toán overlap
    overlap = {x for x in overlap if x != 'unknown'}
    print(f"\n4. Các item_type mà User {sample_uid} vừa View vừa Purchase: {overlap}")

1. Tổng số khách hàng có phát sinh giao dịch mua: 39406
2. Số giao dịch mua bị khuyết thông tin item_type: 39393 / 52825

3. Lịch sử thô của User 225644257:
            event_time event_type  product_id item_type
42 2019-11-08 09:16:23       view     7300471   unknown
43 2019-11-08 09:32:44       view     6801055       hdd
44 2019-11-12 04:13:58       view     6800309       hdd
45 2019-11-12 04:19:31   purchase     6800309       hdd
46 2019-11-13 04:48:06       view     7300444   unknown

4. Các item_type mà User 225644257 vừa View vừa Purchase: {'hdd'}


In [25]:
import pandas as pd
import plotly.express as px

# 1. Trích xuất khách hàng CÓ mua hàng và ép kiểu khóa (Join keys) về chuỗi an toàn
purchasers = df_filtered[df_filtered['event_type'] == 'purchase']['user_id'].unique()
df_buyers = df_filtered[df_filtered['user_id'].isin(purchasers)].copy()

df_buyers['category_id'] = df_buyers['category_id'].astype(str)
df_buyers['user_id'] = df_buyers['user_id'].astype(str)

# 2. Đếm số sản phẩm đã xem TRONG CÙNG 1 CATEGORY_ID
view_counts = df_buyers[df_buyers['event_type'] == 'view'] \
    .groupby(['user_id', 'category_id'])['product_id'] \
    .nunique() \
    .reset_index(name='viewed_count')

# 3. Lấy danh sách mã ngành hàng (category_id) đã được thanh toán
purchased_items = df_buyers[df_buyers['event_type'] == 'purchase'][['user_id', 'category_id']].drop_duplicates()

# 4. Ghép nối chính xác bằng category_id (Khắc phục hoàn toàn lỗi tập rỗng)
df_consideration = pd.merge(purchased_items, view_counts, on=['user_id', 'category_id'], how='inner')

# 5. Nội suy lại tên chữ (item_type) cho biểu đồ
# Tìm tên item_type phổ biến nhất cho mỗi category_id (bỏ qua unknown)
valid_mapping = df_buyers[df_buyers['item_type'] != 'unknown'].groupby('category_id')['item_type'].first()

# Ánh xạ tên. Nếu ngành hàng nào khuyết chữ 100%, dùng 6 số đuôi của ID để hiển thị
df_consideration['category_name'] = df_consideration['category_id'].map(valid_mapping.astype(str)) \
    .fillna('ID: ' + df_consideration['category_id'].str[-6:])

# --- TRỰC QUAN HÓA BẰNG PLOTLY ---

# TRỰC QUAN HÓA 1: Phân phối chung
fig_dist = px.histogram(
    df_consideration, 
    x='viewed_count',
    nbins=30,
    title='Khách hàng thường xem bao nhiêu sản phẩm trước khi chốt đơn?',
    labels={'viewed_count': 'Số lượng sản phẩm khác nhau đã xem', 'count': 'Số lượt chốt đơn'}
)
fig_dist.update_layout(xaxis_range=[0, 30])
fig_dist.show(renderer="iframe")



In [26]:
# TRỰC QUAN HÓA 2: Ngành hàng đắn đo nhiều nhất
avg_consideration = df_consideration.groupby('category_name')['viewed_count'].mean().reset_index()
avg_consideration = avg_consideration.sort_values('viewed_count', ascending=False).head(15)

fig_cat = px.bar(
    avg_consideration,
    x='viewed_count',
    y='category_name',
    orientation='h',
    title='Top 15 loại sản phẩm khách hàng đắn đo/so sánh nhiều nhất',
    labels={'viewed_count': 'Trung bình số sản phẩm đã xem', 'category_name': 'Ngành hàng'}
)
fig_cat.update_yaxes(autorange="reversed")
fig_cat.show(renderer="iframe")